In [0]:
%pip install -q openai
dbutils.library.restartPython()

In [0]:
# =====================================================================
# CELL 2: DEMO SETUP -> an input box (widget) + import the shared agent module
# =====================================================================
# A WIDGET is an input box at the top of the notebook. Type a question there,
# then run Cell 3. No code editing needed during the demo.
import os, sys, html, mlflow
sys.path.append(os.getcwd(),"app")          # make files in this folder importable
import agent_core                     # <- the ONE source of truth for the agents

user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/procintel_agent")   # traces go here

dbutils.widgets.text("question",
    "We plan an open Request for Bids for a $15M road works contract in the Transport sector "
    "under prior review. How long will it take, and what lessons should we consider?",
    "Ask the procurement assistant")
print("✅ ready: type a question in the box at the top, then run Cell 3")

In [0]:
# =====================================================================
# CELL 3: ASK -> run the multi-agent system and show answer + evidence
# =====================================================================
question = dbutils.widgets.get("question")
out = agent_core.ask(question, user=user)          # supervise + verify + log, in one call
v   = out["verification"]

badge  = ("#1a7f37", "✅ VERIFIED") if v["verdict"] == "PASS" else ("#b35900", "⚠️ FLAGGED")
agents = " → ".join(t["tool"] for t in out["trace"]) or "no tools (out of scope)"
answer_html = html.escape(out["answer"] or "").replace("\n", "<br>")   # escape = safe HTML
sources_html = "".join(
    f"<li><code>{s['chunk_id']}</code> [{s['doc_family']}] {html.escape(str(s['title']))[:90]} | "
    f"<i>{html.escape(str(s['section']))[:60]}</i></li>" for s in out["sources"])

displayHTML(f"""
<div style="font-family:sans-serif;max-width:900px">
  <h3 style="margin-bottom:4px">❓ {html.escape(question)}</h3>
  <div style="margin:6px 0;color:#666">
     agents: <b>{agents}</b> &nbsp;|&nbsp; latency: <b>{out['latency_ms']/1000:.1f}s</b> &nbsp;|&nbsp;
     <span style="color:{badge[0]};font-weight:bold">{badge[1]}</span>
     (citations verified: {v['n_verified']}/{v['n_citations']})
  </div>
  <div style="padding:12px;border-left:4px solid {badge[0]};background:rgba(127,127,127,0.08)">{answer_html}</div>
  <details style="margin-top:10px"><summary>📚 Sources retrieved ({len(out['sources'])})</summary>
     <ul>{sources_html}</ul></details>
</div>""")